# 1. Load Data

In [ ]:
import pandas as pd
import numpy as np

def load_data(data_path : str):
    df = pd.read_csv(f"{data_path}")

    return df

def save_data(df: pd.DataFrame, data_path : str):

    return df.to_csv(f"{data_path}", index = False)


DATA_DIR = "data/"
POS_CASH_balance = load_data(DATA_DIR + "POS_CASH_balance.csv")

In [2]:
POS_CASH_balance_copy = POS_CASH_balance.copy()
POS_CASH_balance_copy.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


# 2. Customized One Hot Encoder Function

In [3]:
def OHE(df : pd.DataFrame):
    # Get categorical columns
    categorical_columns = list(df.select_dtypes(include = ['object']))

    # OHE 
    df = pd.get_dummies(df, columns = categorical_columns, dummy_na = False, drop_first = False )

    # Return df
    return df 

# 3. Preprocessing, Feature Engineering and Aggregation

### Preprocessing

In [4]:
POS_CASH_balance_copy = POS_CASH_balance_copy.drop(columns = ["SK_ID_PREV"])
POS_CASH_balance_copy = OHE(POS_CASH_balance_copy)
POS_CASH_balance_copy.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_18960\312675149.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = list(df.select_dtypes(include = ['object']))


,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,SK_DPD,SK_DPD_DEF,NAME_CONTRACT_STATUS_Active,NAME_CONTRACT_STATUS_Amortized debt,NAME_CONTRACT_STATUS_Approved,NAME_CONTRACT_STATUS_Canceled,NAME_CONTRACT_STATUS_Completed,NAME_CONTRACT_STATUS_Demand,NAME_CONTRACT_STATUS_Returned to the store,NAME_CONTRACT_STATUS_Signed,NAME_CONTRACT_STATUS_XNA
0,182943,-31,48.0,45.0,0,0,True,False,False,False,False,False,False,False,False
1,367990,-33,36.0,35.0,0,0,True,False,False,False,False,False,False,False,False
2,397406,-32,12.0,9.0,0,0,True,False,False,False,False,False,False,False,False
3,269225,-35,48.0,42.0,0,0,True,False,False,False,False,False,False,False,False
4,334279,-35,36.0,35.0,0,0,True,False,False,False,False,False,False,False,False


### Feature Engineering

In [5]:
# Ratios
POS_CASH_balance_copy['REMAINING_INSTALMENT_RATIO'] = (POS_CASH_balance_copy['CNT_INSTALMENT_FUTURE'] / POS_CASH_balance_copy['CNT_INSTALMENT'].replace(0, np.nan))
POS_CASH_balance_copy['TOTAL_INSTALMENT'] = POS_CASH_balance_copy['CNT_INSTALMENT_FUTURE'] + POS_CASH_balance_copy['CNT_INSTALMENT']

# L12M, L24M
recent_futures = []

for month in [12, 24]:
    recent = POS_CASH_balance_copy[POS_CASH_balance_copy['MONTHS_BALANCE'].between(-month, 0)]
    agg = recent.groupby("SK_ID_CURR").agg(**{
        f"POS_BALANCE_SK_DPD_MEAN_L{month}M" : ('SK_DPD', 'mean'),
        f"POS_BALANCE_SK_DPD_MAX_L{month}M" : ('SK_DPD', 'max'),
        f"POS_BALANCE_SK_DPD_DEF_MEAN_L{month}M" : ('SK_DPD_DEF', 'mean'),
        f"POS_BALANCE_SK_DPD_DEF_MAX_L{month}M" : ('SK_DPD_DEF', 'max'),
        f"POS_BALANCE_INSTALMENT_COUNT_L{month}M" : ('CNT_INSTALMENT', 'size'),
    })
    recent_futures.append(agg)

recent_futures = pd.concat(recent_futures, axis = 1).reset_index()

# Active
active = []
POS_CASH_balance_copy['IS_ACTIVE'] = POS_CASH_balance_copy['NAME_CONTRACT_STATUS_Active'].astype(int)
for flag, name in [('IS_ACTIVE', 'ACTIVE')]:
    sub = POS_CASH_balance_copy[POS_CASH_balance_copy[flag] == 1]
    agg = sub.groupby("SK_ID_CURR").agg(**{
        f"POS_BALANCE_{name}_SK_DPD_MEAN" : ('SK_DPD', 'mean'),
        f"POS_BALANCE_{name}_SK_DPD_DEF_MEAN" : ('SK_DPD_DEF', 'mean'),
        f"POS_BALANCE_{name}_CNT_INSTALMENT_MEAN" : ('CNT_INSTALMENT', 'mean'),
    })
    active.append(agg)
active = pd.concat(active, axis = 1).reset_index()

# Cancel
completed = []
POS_CASH_balance_copy['IS_COMPLETED'] = POS_CASH_balance_copy['NAME_CONTRACT_STATUS_Completed'].astype(int)
for flag, name in [('IS_COMPLETED', 'COMPLETED')]:
    sub = POS_CASH_balance_copy[POS_CASH_balance_copy[flag] == 1]
    agg = sub.groupby("SK_ID_CURR").agg(**{
        f"POS_BALANCE_{name}_SK_DPD_MEAN" : ('SK_DPD', 'mean'),
        f"POS_BALANCE_{name}_SK_DPD_DEF_MEAN" : ('SK_DPD_DEF', 'mean'),
        f"POS_BALANCE_{name}_CNT_INSTALMENT_MEAN" : ('CNT_INSTALMENT', 'mean'),
    })
    completed.append(agg)
completed = pd.concat(completed, axis = 1).reset_index()

### Aggregation

In [6]:
# Aggregation dict
POS_CASH_balance_copy_numeric_aggregation = {}
POS_CASH_balance_copy_OHE_aggregation = {}


# Numeric
POS_CASH_balance_copy_numeric_aggregation = {
    'MONTHS_BALANCE' : ['count','max','min'],
    'CNT_INSTALMENT' : ['mean', 'max', 'min', 'nunique'],
    'CNT_INSTALMENT_FUTURE' : ['mean', 'max', 'min'],
    'REMAINING_INSTALMENT_RATIO' : ['mean', 'max', 'min'],
    'SK_DPD' : ['mean','max'],
    'SK_DPD_DEF' : ['mean','max'],
    'TOTAL_INSTALMENT' : ['mean','max','min']
}

# OHE Cols
ohe_cols = [col for col in POS_CASH_balance_copy.columns if col.startswith("NAME_CONTRACT_STATUS_")]
POS_CASH_balance_copy_OHE_aggregation = {col : ['mean'] for col in ohe_cols}

# Aggregation
POS_CASH_balance_agg = POS_CASH_balance_copy.groupby("SK_ID_CURR").agg({**POS_CASH_balance_copy_numeric_aggregation, **POS_CASH_balance_copy_OHE_aggregation})
POS_CASH_balance_agg.columns = [f"POS_CASH_{col}_{status}".upper() for col, status in POS_CASH_balance_agg.columns]
POS_CASH_balance_agg = POS_CASH_balance_agg.reset_index()
POS_CASH_balance_agg = POS_CASH_balance_agg.merge(recent_futures, on = 'SK_ID_CURR', how = 'left')
POS_CASH_balance_agg = POS_CASH_balance_agg.merge(active, on = 'SK_ID_CURR', how = 'left')
POS_CASH_balance_agg = POS_CASH_balance_agg.merge(completed, on = 'SK_ID_CURR', how = 'left')
POS_CASH_balance_agg.head()

,SK_ID_CURR,POS_CASH_MONTHS_BALANCE_COUNT,POS_CASH_MONTHS_BALANCE_MAX,POS_CASH_MONTHS_BALANCE_MIN,POS_CASH_CNT_INSTALMENT_MEAN,POS_CASH_CNT_INSTALMENT_MAX,POS_CASH_CNT_INSTALMENT_MIN,POS_CASH_CNT_INSTALMENT_NUNIQUE,POS_CASH_CNT_INSTALMENT_FUTURE_MEAN,POS_CASH_CNT_INSTALMENT_FUTURE_MAX,...,POS_BALANCE_SK_DPD_MAX_L24M,POS_BALANCE_SK_DPD_DEF_MEAN_L24M,POS_BALANCE_SK_DPD_DEF_MAX_L24M,POS_BALANCE_INSTALMENT_COUNT_L24M,POS_BALANCE_ACTIVE_SK_DPD_MEAN,POS_BALANCE_ACTIVE_SK_DPD_DEF_MEAN,POS_BALANCE_ACTIVE_CNT_INSTALMENT_MEAN,POS_BALANCE_COMPLETED_SK_DPD_MEAN,POS_BALANCE_COMPLETED_SK_DPD_DEF_MEAN,POS_BALANCE_COMPLETED_CNT_INSTALMENT_MEAN
0,100001,9,-53,-96,4.000000,4.0,4.0,1,1.444444,4.0,...,NaN,NaN,NaN,NaN,1.0,1.0,4.000000,0.0,0.0,4.0
1,100002,19,-1,-19,24.000000,24.0,24.0,1,15.000000,24.0,...,0.0,0.0,0.0,19.0,0.0,0.0,24.000000,NaN,NaN,NaN
2,100003,28,-18,-77,10.107143,12.0,6.0,3,5.785714,12.0,...,0.0,0.0,0.0,11.0,0.0,0.0,10.384615,0.0,0.0,6.5
3,100004,4,-24,-27,3.750000,4.0,3.0,2,2.250000,4.0,...,0.0,0.0,0.0,1.0,0.0,0.0,4.000000,0.0,0.0,3.0
4,100005,11,-15,-25,11.700000,12.0,9.0,2,7.200000,12.0,...,0.0,0.0,0.0,10.0,0.0,0.0,12.000000,0.0,0.0,9.0


# 4. Save Data

In [ ]:
save_data(POS_CASH_balance_agg, DATA_DIR + "pos_cash_balanced_agg_final.csv")